# Ứng dụng mô hình học máy về hệ thống gợi ý sản phẩm
## Bài toán: gợi ý sản phẩm thời trang tương tự
Dữ liệu lấy từ *Fashion Product Images (Small)* trên Kaggle. Notebook trình bày theo thứ tự: mô tả dữ liệu → kiểm tra và làm sạch → trực quan hóa → tạo đặc trưng → chia train/test → so sánh mô hình → đánh giá → thử gợi ý → lưu mô hình.

Sản phẩm được gợi ý dựa trên đặc trưng nội dung và ảnh, vì bộ dữ liệu không có lịch sử mua hàng của người dùng.

In [ ]:
# Nạp thư viện và hàm dùng chung với ứng dụng web.
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from train_model import CSV_PATH, MODEL_DIR, load_products, train

raw = pd.read_csv(CSV_PATH, on_bad_lines='skip')
print('Kích thước dữ liệu gốc:', raw.shape)
display(raw.head())

## 1. Mô tả dữ liệu
| Cột | Ý nghĩa |
|---|---|
| id | Mã sản phẩm, đồng thời là tên file ảnh |
| gender | Đối tượng sử dụng |
| masterCategory | Nhóm hàng chính |
| subCategory | Danh mục phụ |
| articleType | Loại sản phẩm; dùng làm nhãn đánh giá, không đưa vào đặc trưng |
| baseColour | Màu cơ bản |
| season | Mùa |
| year | Năm sản phẩm |
| usage | Mục đích sử dụng |
| productDisplayName | Tên sản phẩm |

In [ ]:
# Kiểm tra kiểu dữ liệu, giá trị thiếu và bản ghi trùng mã.
raw.info()
display(raw.isna().sum().rename('Giá trị thiếu'))
print('Mã sản phẩm trùng:', raw['id'].duplicated().sum())
display(raw.describe(include='all').T)

## 2. Làm sạch dữ liệu
Bỏ dòng thiếu mã hoặc tên, chỉ giữ một dòng cho mỗi mã, chuẩn hóa giá trị thiếu ở thuộc tính phân loại và chỉ dùng sản phẩm có ảnh tương ứng. Để huấn luyện nhanh trên máy cá nhân, lấy mẫu tái lập được 15.000 sản phẩm từ dữ liệu hợp lệ.

In [ ]:
# Hàm này thực hiện đúng bước làm sạch đang dùng trong ứng dụng web.
products, available_count = load_products(limit=15000)
print('Sản phẩm hợp lệ có ảnh:', available_count)
print('Sản phẩm dùng trong mô hình:', len(products))
display(products[['id','productDisplayName','gender','masterCategory','subCategory','articleType','baseColour','season','year','usage']].head())

In [ ]:
# Hiển thị ảnh thật của một sản phẩm trong bộ dữ liệu.
sample_product = products.iloc[0]
print(sample_product['productDisplayName'])
display(Image(filename=sample_product['image_path']))

## 3. Trực quan hóa và nhận xét dữ liệu
Các biểu đồ dưới đây được tính từ dữ liệu đã làm sạch. Khi viết báo cáo, nên mô tả số liệu hiển thị thực tế thay vì dùng tỷ lệ cố định từ một lần chạy khác.

In [ ]:
# Xem quy mô các nhóm hàng và loại sản phẩm phổ biến.
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
products['masterCategory'].value_counts().plot.barh(ax=axes[0], title='Sản phẩm theo nhóm hàng')
products['articleType'].value_counts().head(15).plot.barh(ax=axes[1], title='15 loại sản phẩm phổ biến')
plt.tight_layout()
plt.show()

In [ ]:
# Xem phân bố màu, mùa và mục đích sử dụng.
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
products['baseColour'].value_counts().head(10).plot.barh(ax=axes[0], title='10 màu phổ biến')
products['season'].value_counts().plot.bar(ax=axes[1], title='Phân bố theo mùa')
products['usage'].value_counts().head(8).plot.barh(ax=axes[2], title='Mục đích sử dụng')
plt.tight_layout()
plt.show()

In [ ]:
# Quan sát năm sản phẩm và độ dài tên sản phẩm.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
products['year'].hist(bins=15, ax=axes[0])
axes[0].set_title('Năm sản phẩm')
products['productDisplayName'].str.len().hist(bins=25, ax=axes[1])
axes[1].set_title('Độ dài tên sản phẩm')
plt.tight_layout()
plt.show()

## 4. Tiền xử lý và tạo đặc trưng
- Tên sản phẩm: TF-IDF với từ đơn và cặp từ.
- Thuộc tính: One-Hot cho đối tượng, nhóm hàng, danh mục phụ, màu, mùa và mục đích sử dụng.
- Năm: chuẩn hóa bằng StandardScaler.
- Ảnh: trung bình RGB và histogram màu cho từng kênh.

Các ma trận được ghép theo trọng số rồi chuẩn hóa để dùng độ tương tự cosine. `articleType` được giữ ngoài đầu vào để đánh giá khả năng gợi ý đúng loại hàng.

## 5. Chia tập huấn luyện và kiểm tra
Chia ngẫu nhiên có `random_state=42` theo tỷ lệ 80/20 và phân tầng theo nhóm hàng chính. Bộ mã hóa chỉ được `fit` trên tập huấn luyện; sản phẩm truy vấn kiểm tra không xuất hiện trong chỉ mục đánh giá.

In [ ]:
# Kiểm tra tỷ lệ chia bằng cùng cấu hình với mã huấn luyện.
from sklearn.model_selection import train_test_split
indexes = np.arange(len(products))
train_indexes, test_indexes = train_test_split(indexes, test_size=0.2, random_state=42, stratify=products['masterCategory'])
print('Train:', len(train_indexes), 'sản phẩm')
print('Test:', len(test_indexes), 'sản phẩm')
print('Trùng chỉ mục:', len(set(train_indexes) & set(test_indexes)))

## 6. Huấn luyện và so sánh mô hình
So sánh ba cách biểu diễn: chỉ tên, chỉ thuộc tính cấu trúc và kết hợp tên + thuộc tính + năm + màu ảnh. Trong bài toán gợi ý, bộ huấn luyện là các sản phẩm trong chỉ mục; bộ kiểm tra gồm các sản phẩm làm truy vấn mới.

In [ ]:
# Huấn luyện, đánh giá và lưu mô hình kết hợp để giao diện sử dụng.
metrics = train(limit=15000)
print(json.dumps(metrics, ensure_ascii=False, indent=2))

## 7. Đánh giá kết quả
**Precision@10** là tỷ lệ sản phẩm cùng `articleType` trong 10 gợi ý đầu, trung bình trên các truy vấn. **HitRate@10** là tỷ lệ truy vấn có ít nhất một gợi ý cùng loại. Nhãn `articleType` không tham gia tạo đặc trưng. Các chỉ số này đo độ phù hợp về loại hàng, không đo xác suất mua.

In [ ]:
# So sánh kết quả thật của ba mô hình, không điền số liệu thủ công.
comparison = pd.DataFrame(metrics['comparison']).T
display(comparison)
comparison[['precision_at_10','hit_rate_at_10']].plot.bar(figsize=(9, 4), ylim=(0, 1), title='So sánh các mô hình gợi ý')
plt.xticks(rotation=0)
plt.ylabel('Điểm đánh giá')
plt.tight_layout()
plt.show()

## 8. Thử gợi ý một sản phẩm
Sau khi đánh giá xong, mô hình kết hợp được học lại trên toàn bộ 15.000 sản phẩm để phục vụ ứng dụng. Ví dụ dưới đây loại sản phẩm được chọn khỏi kết quả.

In [ ]:
# Tải mô hình đã lưu và lấy 10 hàng xóm gần nhất.
saved = pd.read_pickle(MODEL_DIR / 'products.pkl')
bundle = joblib.load(MODEL_DIR / 'fashion_model.joblib')
distances, neighbors = bundle['model'].kneighbors(bundle['matrix'][[0]], n_neighbors=11)
pairs = [(index, distance) for index, distance in zip(neighbors[0], distances[0]) if index != 0][:10]
top_indexes = [index for index, _ in pairs]
result = saved.iloc[top_indexes][['id','productDisplayName','articleType','baseColour','season','usage']].copy()
result['Độ tương tự'] = [1 - distance for _, distance in pairs]
display(result)
display(Image(filename=saved.iloc[top_indexes[0]]['image_path']))

## 9. Lưu mô hình và kết luận
`models/fashion_model.joblib` chứa mô hình và ma trận đặc trưng; `models/products.pkl` chứa danh mục sản phẩm; `models/metrics.json` và `models/model_comparison.csv` chứa kết quả đánh giá. Ứng dụng Streamlit đọc các tệp này để hiển thị gợi ý.

Giới hạn: dữ liệu không có người dùng và giao dịch, vì vậy không thể kiểm chứng khả năng mua hàng hoặc gợi ý cá nhân hóa dài hạn. Có thể mở rộng sau bằng lịch sử tương tác người dùng.

In [ ]:
# Xác nhận các tệp đầu ra đã được lưu thành công.
for filename in ['fashion_model.joblib','products.pkl','metrics.json','model_comparison.csv']:
    path = MODEL_DIR / filename
    print(filename, '→', 'Đã lưu' if path.exists() else 'Chưa có')